In [1]:
import pickle
import numpy as np
from transformers import BartTokenizer, BartForConditionalGeneration
from data import ZuCo_dataset
from torch.utils.data import DataLoader

whole_dataset_dicts = []

dataset_path_task1 = '/home/saul_park/workspace/code/Baseline/dataset/ZuCo/task1-SR/pickle/task1-SR-dataset.pickle'
with open(dataset_path_task1, 'rb') as handle:
    whole_dataset_dicts.append(pickle.load(handle))

dataset_path_task2 = '/home/saul_park/workspace/code/Baseline/dataset/ZuCo/task2-NR/pickle/task2-NR-dataset.pickle'
with open(dataset_path_task2, 'rb') as handle:
    whole_dataset_dicts.append(pickle.load(handle))


dataset_path_taskNRv2 = '/home/saul_park/workspace/code/Baseline/dataset/ZuCo/task2-NR-2.0/pickle/task2-NR-2.0-dataset.pickle'

with open(dataset_path_taskNRv2, 'rb') as handle:
    whole_dataset_dicts.append(pickle.load(handle))

tokenizer = BartTokenizer.from_pretrained('facebook/bart-large')
subject_choice = 'ALL'
batch_size = 32
eeg_type_choice = 'GD'
bands_choice = ['_t1','_t2','_a1','_a2','_b1','_b2','_g1','_g2']
dataset_setting = 'unique_sent'
train_input = 'EEG'

# train dataset
train_set = ZuCo_dataset(whole_dataset_dicts, 'train', tokenizer, subject = subject_choice, eeg_type = eeg_type_choice, bands = bands_choice, setting = dataset_setting, test_input=train_input)
# dev dataset
dev_set = ZuCo_dataset(whole_dataset_dicts, 'dev', tokenizer, subject = subject_choice, eeg_type = eeg_type_choice, bands = bands_choice, setting = dataset_setting, test_input=train_input)
# test dataset
test_set = ZuCo_dataset(whole_dataset_dicts, 'test', tokenizer, subject = subject_choice, eeg_type = eeg_type_choice, bands = bands_choice, setting = dataset_setting)
dataset_sizes = {'train': len(train_set), 'dev': len(dev_set)}
print('[INFO]train_set size: ', len(train_set))
print('[INFO]dev_set size: ', len(dev_set))
print('[INFO]test_set size: ', len(test_set))

# train dataloader
train_dataloader = DataLoader(train_set, batch_size = batch_size, shuffle=True, num_workers=4)
# dev dataloader
val_dataloader = DataLoader(dev_set, batch_size = 1, shuffle=False, num_workers=4)
test_dataloader = DataLoader(test_set, batch_size = 1, shuffle=False, num_workers=4)
# dataloaders
dataloaders = {'train':train_dataloader, 'dev':val_dataloader, 'test':test_dataloader}

/tmp/ipykernel_1098723/3267645407.py:11: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  whole_dataset_dicts.append(pickle.load(handle))
/tmp/ipykernel_1098723/3267645407.py:15: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public Num

[INFO]loading 3 task datasets
[INFO]using subjects:  ['ZAB', 'ZDM', 'ZDN', 'ZGW', 'ZJM', 'ZJN', 'ZJS', 'ZKB', 'ZKH', 'ZKW', 'ZMG', 'ZPH']
train divider = 320
dev divider = 360
[INFO]initializing a train set...
++ adding task to dataset, now we have: 3609
[INFO]using subjects:  ['ZAB', 'ZDM', 'ZDN', 'ZGW', 'ZJM', 'ZJN', 'ZJS', 'ZKB', 'ZKH', 'ZKW', 'ZMG', 'ZPH']
train divider = 240
dev divider = 270
[INFO]initializing a train set...
++ adding task to dataset, now we have: 6254
[INFO]using subjects:  ['YDR', 'YFR', 'YFS', 'YAC', 'YDG', 'YHS', 'YMD', 'YLS', 'YRH', 'YRK', 'YSD', 'YRP', 'YSL', 'YTL', 'YAG', 'YAK', 'YIS', 'YMS']
train divider = 279
dev divider = 313
[INFO]initializing a train set...
++ adding task to dataset, now we have: 10710
[INFO]input tensor size: torch.Size([56, 840])

[INFO]loading 3 task datasets
[INFO]using subjects:  ['ZAB', 'ZDM', 'ZDN', 'ZGW', 'ZJM', 'ZJN', 'ZJS', 'ZKB', 'ZKH', 'ZKW', 'ZMG', 'ZPH']
train divider = 320
dev divider = 360
[INFO]initializing a dev set

In [2]:
for input_embeddings, seq_len, input_masks, input_mask_invert, target_ids, target_mask, rawEEG in dataloaders['test']:
    break

input_embeddings_batch = input_embeddings.to('cpu').float() # B, 56, 840
# input_embeddings_batch = sent_level_EEG.unsqueeze(1).to(device).float()
input_masks_batch = input_masks.to('cpu') # B, 56
target_ids_batch = target_ids.to('cpu') # B, 56
input_mask_invert_batch = input_mask_invert.to('cpu') # B, 56

In [29]:
import torch
from ldm.diffusion_module import register_schedule
from ldm.util import q_sample, extract_into_tensor, checkpoint

device = 'cpu'
schedule = register_schedule(device = device)

noise = torch.randn_like(input_embeddings_batch, device=device)
timesteps = torch.randint(0, 1000, (input_embeddings_batch.shape[0],), device=device).long() # (B,)
x_noisy = q_sample(input_embeddings_batch, timesteps, schedule['sqrt_alphas_cumprod'], schedule['sqrt_one_minus_alphas_cumprod'], noise=noise) # (B, 1024, 8)
print(timesteps.dtype) # (B,)

torch.int64


/home/saul_park/workspace/code/pretrain_eeg_encoder/ldm/diffusion_module.py:39: RuntimeWarning: divide by zero encountered in divide
  lvlb_weights = betas ** 2 / (


In [26]:
T = 1000
torch.full((input_embeddings_batch.shape[0],), T, dtype=torch.long, device='cpu')


tensor([1000])

In [21]:
target_ids_batch.shape

torch.Size([1, 56])

In [20]:
timesteps

tensor([209])

In [19]:
import torch
from ldm.diffusion_module import register_schedule
from ldm.util import q_sample, extract_into_tensor, checkpoint

device = 'cpu'
schedule = register_schedule(device = device)

noise = torch.randn(target_ids_batch.shape, device=target_ids_batch.device)
timesteps = torch.randint(0, 1000, (target_ids_batch.shape[0],), device=device).long() # (B,)
x_noisy = q_sample(target_ids_batch, timesteps, schedule['sqrt_alphas_cumprod'], schedule['sqrt_one_minus_alphas_cumprod'], noise=noise) # (B, 1024, 8)
x_noisy_int = x_noisy.long().clamp(min=0, max=tokenizer.vocab_size - 1)  # 범위 제한
print("Original target IDs:", target_ids_batch)
print("Noisy target IDs:", x_noisy_int)

Original target IDs: tensor([[    0, 15173,    63,  1270, 19659,     6,    10,  2526,    12, 25268,
          1846,  4149, 20231,    66,    31,     5, 16055,  5317,  7171,  6559,
           516,     4,     2,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1]])
Noisy target IDs: tensor([[    0, 14205,    58,  1188, 18404,     5,     8,  2365,    11, 23656,
          1727,  3883, 18940,    62,    28,     4, 15030,  4977,  6713,  6140,
           482,     4,     1,     1,     0,     1,     0,     0,     1,     1,
             1,     0,     1,     0,     1,     0,     0,     0,     1,     1,
             0,     0,     0,     1,     0,     1,     0,     1,     0,     0,
             1,     1,     0,     0,     0,     1]])


In [17]:
print('original: ', tokenizer.batch_decode(target_ids_batch, skip_special_tokens=True)[0])
print('noisy: ', tokenizer.batch_decode(x_noisy_int, skip_special_tokens=True)[0])

original:  Everything its title implies, a standard-issue crime drama spat out from the Tinseltown assembly line.
noisy:   turn. AZ government leadership him best charged. Lely health media:
